In [129]:
import warnings
warnings.simplefilter("ignore")
import numpy as np
import pandas as pd 
from lightkurve import LightCurve
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve 
import lightkurve as lk

for att in ['axes.labelsize', 'axes.titlesize', 'legend.fontsize',
            'legend.fontsize', 'xtick.labelsize', 'ytick.labelsize']:
    plt.rcParams[att] = 15

search_result = search_lightcurve("AU Mic") 
#print(search_result)
lc2min = search_result[4].download() 
lc2 = lc2min.remove_nans().remove_outliers()
lc2 = lc2[lc2.quality == 0]
lc2_m = lc2.normalize().remove_nans()

x2min = np.ascontiguousarray(lc2_m.time.value, dtype=np.float64) 
y2min = np.ascontiguousarray(lc2_m.flux, dtype=np.float64)
yerr2min = np.ascontiguousarray(lc2_m.flux_err, dtype=np.float64)
lcquality = np.ascontiguousarray(lc2_m.quality, dtype=np.float64)

In [130]:

# ============================================================
# PASSO 2: CARREGAR MÁSCARA DE FLARES E TRÂNSITOS (TXT EDITÁVEL)
# ============================================================
print("\n" + "="*60)
print("ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do TXT")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# Arquivo editável com colunas: t_ini, t_fim
txt_path = "intervalos_flares_transitos_editavel.txt"

try:
    # Lê como CSV simples (separado por vírgula), mas em arquivo .txt
    df_intervalos = pd.read_csv(txt_path, sep=",", comment="#")

    # Validação básica das colunas esperadas
    colunas_esperadas = {"t_ini", "t_fim"}
    if not colunas_esperadas.issubset(df_intervalos.columns):
        raise ValueError(f"Arquivo deve conter as colunas {colunas_esperadas}. Colunas encontradas: {set(df_intervalos.columns)}")

    # Converte para lista de pares [inicio, fim]
    mascara_flares_list = df_intervalos[["t_ini", "t_fim"]].dropna().values.tolist()

    print(f"✓ {len(mascara_flares_list)} intervalo(s) carregado(s) de '{txt_path}':")
    for i, (ini, fim) in enumerate(mascara_flares_list, start=1):
        print(f"  {i}. [{ini:.6f}, {fim:.6f}]")

except Exception as e:
    print(f"✗ Erro ao carregar '{txt_path}': {e}")
    print("→ Usando lista vazia.")
    mascara_flares_list = []

# Criar máscara booleana
mascara_flares = np.zeros(len(t), dtype=bool)
for ini, fim in mascara_flares_list:
    mascara_flares |= (t >= ini) & (t <= fim)

mask_good_flares = ~mascara_flares

print(f"\nMáscara criada:")
print(f"  Pontos excluídos (flares/trânsitos): {mascara_flares.sum()}")
print(f"  Pontos para ajuste: {mask_good_flares.sum()}")



ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do TXT
✓ 25 intervalo(s) carregado(s) de 'intervalos_flares_transitos_editavel.txt':
  1. [3883.239461, 3883.334868]
  2. [3884.751868, 3884.843937]
  3. [3885.024950, 3885.109542]
  4. [3885.549959, 3885.617899]
  5. [3886.130183, 3886.411659]
  6. [3888.081298, 3888.115210]
  7. [3888.784350, 3888.805991]
  8. [3889.179949, 3889.251286]
  9. [3889.826802, 3889.854535]
  10. [3891.013277, 3891.076262]
  11. [3891.741181, 3891.795356]
  12. [3891.844004, 3891.908792]
  13. [3891.950028, 3892.001562]
  14. [3892.631057, 3892.697367]
  15. [3892.812488, 3892.900441]
  16. [3893.017865, 3893.101673]
  17. [3893.403291, 3893.458281]
  18. [3896.687407, 3896.849926]
  19. [3897.567644, 3897.779646]
  20. [3899.153300, 3899.218797]
  21. [3899.542373, 3899.616046]
  22. [3900.664900, 3900.748400]
  23. [3904.512267, 3904.614076]
  24. [3904.782061, 3904.847219]
  25. [3905.340994, 3905.482584]

Máscara criada:
  Pontos excluídos (flares/t

In [131]:
from astropy.stats import sigma_clip
import numpy as np
import matplotlib.pyplot as plt
from lightkurve import LightCurve

# ============================================================
# PASSO 3: AJUSTE POLINOMIAL + FLATTEN LOCAL
# ============================================================

print("\n" + "="*60)
print("AJUSTE: Polinomial com Segmentos e Costura de Bordas")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# ============================================================
# REGIÕES MANUAIS
# ============================================================

ajustes_manuais = [

    [3883.0050, 3883.4554, "poly",    4, 2.5],
    [3883.1409, 3883.7849, "poly",    4, 2.5],
    [3883.8264, 3884.0955, "poly",    4, 2.5],
    [3884.2828, 3885.3000, "poly",    4, 2.5],
    [3885.2839, 3885.4326, "poly",    4, 2.5],
    [3885.6205, 3885.7794, "poly",    2, 2.5],
    
    [3892.6280, 3892.7040, "poly",    1, 2.5],


    #[3887.1853, 3888.4396, "flatten", None, None],
]

# ============================================================
# FLATTEN GLOBAL
# ============================================================

flcd, trend = lc2_m.flatten(
    window_length=320,
    polyorder=3,
    return_trend=True,
    break_tolerance=10,
    niters=4,
    sigma=2.5,
    mask=mask_good_flares
)

# ============================================================
# AJUSTE AUTOMÁTICO
# ============================================================

N_SEGMENTOS_AUTO = 40
GRAU_AUTO = 4
SIGMA_AUTO = 2.9

# ============================================================
# FUNÇÕES
# ============================================================

def fitting_segment(t_seg, f_seg, mask_seg, deg, sigma):

    if len(t_seg) < deg + 2:
        return np.full_like(f_seg, np.nanmedian(f_seg))

    t_mid = np.median(t_seg)
    t_s = t_seg - t_mid

    good = mask_seg.copy()

    modelo = np.full_like(f_seg, np.nan)

    for _ in range(5):

        if np.sum(good) < deg + 2:
            break

        coef = np.polyfit(
            t_s[good],
            f_seg[good],
            deg=deg
        )

        modelo_good = np.polyval(
            coef,
            t_s[good]
        )

        modelo = np.interp(
            t_s,
            t_s[good],
            modelo_good
        )

        resid = f_seg - modelo

        clipped = sigma_clip(
            resid[good],
            sigma=sigma,
            maxiters=1
        )

        if clipped.mask is np.ma.nomask:
            break

        good[np.where(good)[0]] = ~clipped.mask

    if np.all(np.isnan(modelo)):
        modelo = np.full_like(
            f_seg,
            np.nanmedian(f_seg)
        )

    return modelo


def costurar_bordas(
    t,
    modelo,
    bordas,
    tamanho_janela=20
):

    modelo_costurado = np.copy(modelo)

    print(
        f"\nAplicando costura em "
        f"{len(bordas)} bordas..."
    )

    for borda_idx in bordas:

        inicio = max(
            0,
            borda_idx - tamanho_janela
        )

        fim = min(
            len(t)-1,
            borda_idx + tamanho_janela
        )

        if fim <= inicio:
            continue

        fluxo_A = modelo[inicio]
        fluxo_B = modelo[fim]

        tempo_A = t[inicio]
        tempo_B = t[fim]

        tempos = t[inicio:fim+1]

        transicao = np.interp(
            tempos,
            [tempo_A, tempo_B],
            [fluxo_A, fluxo_B]
        )

        modelo_costurado[inicio:fim+1] = transicao

    return modelo_costurado

# ============================================================
# AJUSTE AUTOMÁTICO BASE
# ============================================================

modelo_manchas = np.zeros_like(f)

bordas_indices = set()

edges = np.linspace(
    t.min(),
    t.max(),
    N_SEGMENTOS_AUTO + 1
)

segmentos = [
    (edges[i], edges[i+1])
    for i in range(N_SEGMENTOS_AUTO)
]

print("Ajuste automático...")

for i, (ini, fim) in enumerate(segmentos):

    idx = (t >= ini) & (t <= fim)

    if not np.any(idx):
        continue

    modelo_manchas[idx] = fitting_segment(
        t[idx],
        f[idx],
        mask_good_flares[idx],
        GRAU_AUTO,
        SIGMA_AUTO
    )

    if i > 0:
        bordas_indices.add(
            np.where(idx)[0][0]
        )

# ============================================================
# AJUSTES MANUAIS
# ============================================================

print(
    f"Aplicando "
    f"{len(ajustes_manuais)} ajustes manuais..."
)

for ini, fim, metodo, grau, sig in ajustes_manuais:

    idx = (t >= ini) & (t <= fim)

    if not np.any(idx):
        continue

    if metodo == "poly":

        modelo_manchas[idx] = fitting_segment(
            t[idx],
            f[idx],
            mask_good_flares[idx],
            grau,
            sig
        )

    elif metodo == "flatten":

        print(
            f"USANDO FLATTEN "
            f"{ini:.4f} - {fim:.4f}"
        )

        modelo_manchas[idx] = trend.flux.value[idx]

        idx_inicio = np.where(idx)[0][0]

        janela = 15

        i0 = max(
            0,
            idx_inicio - janela
        )

        i1 = min(
            len(modelo_manchas)-1,
            idx_inicio + janela
        )

        modelo_manchas[i0:i1+1] = np.linspace(
            modelo_manchas[i0],
            modelo_manchas[i1],
            i1 - i0 + 1
        )

    bordas_indices.add(
        np.where(idx)[0][0]
    )

    bordas_indices.add(
        np.where(idx)[0][-1]
    )

# ============================================================
# COSTURA FINAL
# ============================================================

modelo_manchas_suave = costurar_bordas(
    t,
    modelo_manchas,
    sorted(bordas_indices),
    tamanho_janela=30
)

# ============================================================
# INSPEÇÃO DO TRECHO PROBLEMÁTICO
# ============================================================

idx_print = (
    (t >= 3887.15) &
    (t <= 3887.25)
)

print("\nTRECHO 3887.15 - 3887.25\n")

for tempo, valor in zip(
    t[idx_print],
    modelo_manchas_suave[idx_print]
):
    print(
        f"{tempo:.10f}    "
        f"{valor:.10f}"
    )

# ============================================================
# RESÍDUO
# ============================================================

residual_manchas = (
    f /
    modelo_manchas_suave
)

print("\nOK Ajuste concluído.")

# ============================================================
# GRÁFICOS
# ============================================================

%matplotlib qt

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(12,10)
)

ax1.plot(
    t,
    f,
    'k.-',
    ms=1.5,
    lw=0.5,
    alpha=0.6,
    label='Dados'
)

ax1.plot(
    t,
    modelo_manchas_suave,
    'r-',
    lw=2.5,
    label='Modelo'
)

ax1.set_ylabel("Fluxo")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(
    t,
    residual_manchas,
    'b.-',
    ms=1.5,
    lw=0.5,
    alpha=0.7
)

ax2.axhline(
    1,
    ls='--',
    alpha=0.5
)

ax2.set_ylabel("Residual")
ax2.set_xlabel("Tempo [BTJD]")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


AJUSTE: Polinomial com Segmentos e Costura de Bordas
Ajuste automático...
Aplicando 7 ajustes manuais...

Aplicando costura em 50 bordas...

TRECHO 3887.15 - 3887.25

3887.1512388559    1.0019648075
3887.1526277395    1.0020159483
3887.1540166231    1.0020672083
3887.1554055071    1.0021185875
3887.1567943907    1.0021698475
3887.1581832742    1.0022212267
3887.1595721583    1.0022726059
3887.1609610418    1.0023241043
3887.1623499254    1.0023754835
3887.1637388094    1.0024269819
3887.1651276930    1.0024784803
3887.1665165766    1.0025299788
3887.1679054601    1.0025815964
3887.1692943442    1.0026332140
3887.1706832277    1.0026848316
3887.1720721113    1.0027364492
3887.1734609949    1.0027881861
3887.1748498785    1.0028399229
3887.1762387625    1.0028916597
3887.1776276461    1.0029433966
3887.1790165296    1.0029952526
3887.1804054132    1.0030469894
3887.1817942968    1.0030988455
3887.1831831803    1.0031508207
3887.1845720639    1.0032026768
3887.1859609479    1.0032546520


In [36]:
%matplotlib qt
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# PARAMETROS DOS PLANETAS (Plavchan et al. 2020)
# ============================================================

# AU Mic b
T0_b    = 1330.39051   # TBJD (BJD - 2457000)
P_b     = 8.463000     # dias
dur_b   = 3.50 / 24.0  # duracao em dias

# AU Mic c
T0_c    = 1342.2223    # TBJD
P_c     = 18.859019    # dias
dur_c   = 4.5  / 24.0  # duracao em dias

# ============================================================
# FUNCAO: centros de transito no intervalo de t
# ============================================================

def transit_times(T0, P, t_min, t_max):
    n_min = int(np.ceil( (t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    return [T0 + n * P for n in range(n_min, n_max + 1)]

t_min, t_max = t.min(), t.max()
centers_b = transit_times(T0_b, P_b, t_min, t_max)
centers_c = transit_times(T0_c, P_c, t_min, t_max)

print(f'Transitos AU Mic b: {len(centers_b)}')
for tc in centers_b: print(f'  BTJD = {tc:.5f}')
print(f'Transitos AU Mic c: {len(centers_c)}')
for tc in centers_c: print(f'  BTJD = {tc:.5f}')

# ============================================================
# FIGURA: 2 paineis
# ============================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# ---- Painel superior: dados + modelo ----
ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.5, label='Dados Originais', zorder=1)
ax1.plot(t, modelo_manchas_suave, 'r-', lw=2, label='Modelo (manchas)', zorder=3)

# Mascara de flares/transitos (CSV)
_lm = False
for ini, fim in mascara_flares_list:
    lbl = 'Flares e Transitos detectados (IV)' if not _lm else '_nolegend_'
    ax1.axvspan(ini, fim, color='orange', alpha=0.25, zorder=2, label=lbl)
    _lm = True

# Transitos AU Mic b
_lb = False
for tc in centers_b:
    lbl = 'Transito AU Mic b' if not _lb else '_nolegend_'
    ax1.axvspan(tc - dur_b/2, tc + dur_b/2, color='royalblue', alpha=0.28, zorder=2, label=lbl)
    ax1.axvline(tc, color='royalblue', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lb = True

# Transitos AU Mic c
_lc = False
for tc in centers_c:
    lbl = 'Transito AU Mic c' if not _lc else '_nolegend_'
    ax1.axvspan(tc - dur_c/2, tc + dur_c/2, color='seagreen', alpha=0.26, zorder=2, label=lbl)
    ax1.axvline(tc, color='seagreen', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lc = True

ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('AU Mic - Ajuste de Manchas, Mascara e Transitos Planetarios', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.25)

# ---- Painel inferior: residual ----
ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.6, label='Residual', zorder=1)
ax2.axhline(1.0, color='gray', ls='--', lw=1.2, alpha=0.6)

_lm2 = False
for ini, fim in mascara_flares_list:
    lbl = 'Mascara' if not _lm2 else '_nolegend_'
    ax2.axvspan(ini, fim, color='orange', alpha=0.25, zorder=2, label=lbl)
    _lm2 = True

_lb2 = False
for tc in centers_b:
    lbl = 'AU Mic b' if not _lb2 else '_nolegend_'
    ax2.axvspan(tc - dur_b/2, tc + dur_b/2, color='royalblue', alpha=0.28, zorder=2, label=lbl)
    ax2.axvline(tc, color='royalblue', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lb2 = True

_lc2 = False
for tc in centers_c:
    lbl = 'AU Mic c' if not _lc2 else '_nolegend_'
    ax2.axvspan(tc - dur_c/2, tc + dur_c/2, color='seagreen', alpha=0.26, zorder=2, label=lbl)
    ax2.axvline(tc, color='seagreen', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lc2 = True

ax2.set_ylabel('Fluxo Residual', fontsize=14)
ax2.set_xlabel('Tempo [BTJD dias]', fontsize=14)
ax2.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.25)

plt.tight_layout()
plt.show()

Transitos AU Mic b: 3
  BTJD = 3886.21651
  BTJD = 3894.67951
  BTJD = 3903.14251
Transitos AU Mic c: 2
  BTJD = 3888.18986
  BTJD = 3907.04888


In [7]:


%matplotlib qt
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Gráfico 1: Dados originais + manchas
ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados Originais')
ax1.plot(t, modelo_manchas, 'r-', linewidth=2.5, label='Ajuste (Manchas)')
ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('Passo 1: Ajuste de Manchas Estelares', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=12)
ax1.grid(alpha=0.3)





In [8]:
# ============================================================
# PASSO 3: VISUALIZAR RESIDUAIS E SELECIONAR FLARES INTERATIVAMENTE
# ============================================================
print("\n" + "="*60)
print("VISUALIZAÇÃO: Residuais após manchas (para seleção de flares)")
print("="*60)

%matplotlib qt
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Gráfico 1: Dados originais + manchas
ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados Originais')
ax1.plot(t, modelo_manchas, 'r-', linewidth=2.5, label='Ajuste (Manchas)')
ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('Passo 1: Ajuste de Manchas Estelares', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=12)
ax1.grid(alpha=0.3)

# Gráfico 2: Residuais para detecção de flares
ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7, label='Residual (original/manchas)')
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('Fluxo Residual', fontsize=14)
ax2.set_xlabel('Tempo [BTJD dias]', fontsize=14)
ax2.set_title('Residuais para Seleção de Flares e Trânsitos', fontsize=15, fontweight='bold')
ax2.legend(loc='upper right', fontsize=12)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n→ Feche o gráfico e continue na próxima célula para inserir os intervalos de flares/trânsitos")



VISUALIZAÇÃO: Residuais após manchas (para seleção de flares)

→ Feche o gráfico e continue na próxima célula para inserir os intervalos de flares/trânsitos


In [9]:
# ============================================================
# MEDIA E DESVIO PADRAO COM MASCARA
# ============================================================
from astropy.stats import sigma_clipped_stats

# ---- Parâmetros AU Mic d (definidos aqui pois célula 4 não tem) ----
T0_d   = 2458333.3211 - 2457000   # = 1333.3211 TBJD
P_d    = 12.73596                  # dias
dur_d  = None                      # duração não disponível

def transit_times(T0, P, t_min, t_max):
    n_min = int(np.ceil( (t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    return [T0 + n * P for n in range(n_min, n_max + 1)]

centers_d = transit_times(T0_d, P_d, t.min(), t.max())
print(f"Trânsitos AU Mic d: {len(centers_d)}")
for tc in centers_d: print(f"  BTJD = {tc:.5f}")

# ---- Passo 1: filtrar pontos bons ----
x_detrend = []
y_detrend = []
for j in range(len(mask_good_flares)):
    if mask_good_flares[j] == True:
        y_detrend.append(residual_manchas[j])
        x_detrend.append(t[j])

mascara2 = [False] * len(mask_good_flares)

x_detrend = np.array(x_detrend)
y_detrend = np.array(y_detrend)

print(f"\nPontos totais:     {len(t)}")
print(f"Pontos bons:       {len(y_detrend)}")
print(f"Pontos mascarados: {len(t) - len(y_detrend)}")

# ---- Passo 2: média e desvio padrão iniciais ----
y_mean  = np.mean(y_detrend)
y_mean2 = [y_mean] * len(y_detrend)
desvio  = np.std(y_detrend)

med_desv2 = y_mean + 2 * desvio
med_desv3 = y_mean - 2 * desvio

print(f"\n--- 1ª iteração ---")
print(f"Média:                {y_mean:.6f}")
print(f"Desvio padrão:        {desvio:.6f}")
print(f"Corte superior (+2σ): {med_desv2:.6f}")
print(f"Corte inferior (-2σ): {med_desv3:.6f}")

# ---- Passo 3: corte em ±2σ ----
xdefinitivo = []
ydefinitivo = []
for j in range(len(y_detrend)):
    if y_detrend[j] < med_desv2 and y_detrend[j] > med_desv3:
        ydefinitivo.append(y_detrend[j])
        xdefinitivo.append(x_detrend[j])

xdefinitivo = np.array(xdefinitivo)
ydefinitivo = np.array(ydefinitivo)

print(f"\nPontos após corte ±2σ: {len(ydefinitivo)}")

# ---- Passo 4: sigma-clipping iterativo ----
y_m, _, desvio2 = sigma_clipped_stats(ydefinitivo, sigma=3, maxiters=5)
med2     = y_m + 3 * desvio2
med2_inf = y_m - 3 * desvio2

print(f"\n--- 2ª iteração (sigma-clipping 3σ iterativo) ---")
print(f"Média clipped:  {y_m:.6f}")
print(f"Desvio padrão:  {desvio2:.6f}")
print(f"Limiar +3σ:     {med2:.6f}")
print(f"Limiar -3σ:     {med2_inf:.6f}")

# ---- Passo 5: gráfico ----
%matplotlib qt

fig, ax = plt.subplots(1, 1, figsize=(16, 5))

# Residual completo
l1, = ax.plot(t, residual_manchas, 'k.-', ms=1.5, lw=0.5, alpha=0.5, zorder=1)

# Pontos usados no cálculo
l2, = ax.plot(xdefinitivo, ydefinitivo, 'b.', ms=1.5, alpha=0.6, zorder=2)

# Linhas de referência
l3 = ax.axhline(y_m,       color='green',  lw=1.5, ls='-',  alpha=0.9, zorder=5)
l4 = ax.axhline(med2,      color='red',    lw=1.5, ls='--', alpha=0.9, zorder=5)
l5 = ax.axhline(med2_inf,  color='red',    lw=1.5, ls=':',  alpha=0.9, zorder=5)
l6 = ax.axhline(med_desv2, color='orange', lw=1.2, ls='--', alpha=0.8, zorder=5)
l7 = ax.axhline(med_desv3, color='orange', lw=1.2, ls=':',  alpha=0.8, zorder=5)

# Máscara do CSV
p_mask = None
for ini, fim in mascara_flares_list:
    p_mask = ax.axvspan(ini, fim, color='orange', alpha=0.20, zorder=0)

# Trânsitos AU Mic b
p_b = None
for tc in centers_b:
    p_b = ax.axvspan(tc - dur_b/2, tc + dur_b/2, color='royalblue', alpha=0.18, zorder=0)
    ax.axvline(tc, color='royalblue', lw=1.0, ls='--', alpha=0.6)

# Trânsitos AU Mic c
p_c = None
for tc in centers_c:
    p_c = ax.axvspan(tc - dur_c/2, tc + dur_c/2, color='seagreen', alpha=0.18, zorder=0)
    ax.axvline(tc, color='seagreen', lw=1.0, ls='--', alpha=0.6)

# Trânsitos AU Mic d (só linha vertical - sem duração)
l_d = None
for tc in centers_d:
    l_d = ax.axvline(tc, color='purple', lw=1.2, ls='-.', alpha=0.7)

# Legenda manual
legend_handles = [l1, l2, l3, l4, l5, l6, l7]
legend_labels  = [
    'Residual completo',
    'Pontos usados (fora máscara, ±2σ)',
    f'Média = {y_m:.5f}',
    f'+3σ  = {med2:.5f}',
    f'-3σ  = {med2_inf:.5f}',
    f'+2σ corte = {med_desv2:.5f}',
    f'-2σ corte = {med_desv3:.5f}',
]

if p_mask is not None:
    legend_handles.append(p_mask)
    legend_labels.append('Máscara CSV')
if p_b is not None:
    legend_handles.append(p_b)
    legend_labels.append('AU Mic b')
if p_c is not None:
    legend_handles.append(p_c)
    legend_labels.append('AU Mic c')
if l_d is not None:
    legend_handles.append(l_d)
    legend_labels.append('AU Mic d')

ax.legend(legend_handles, legend_labels,
          loc='upper right', fontsize=9, framealpha=0.95, ncol=2)

ax.set_xlabel('Tempo - 2457000 [BTJD dias]', fontsize=13)
ax.set_ylabel('Fluxo Residual Normalizado', fontsize=13)
ax.set_title('AU Mic – Residual com Máscara, Limiares e Trânsitos', fontsize=14, fontweight='bold')
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

Trânsitos AU Mic d: 2
  BTJD = 3893.24906
  BTJD = 3905.98502

Pontos totais:     15111
Pontos bons:       13186
Pontos mascarados: 1925

--- 1ª iteração ---
Média:                1.000026
Desvio padrão:        0.000514
Corte superior (+2σ): 1.001054
Corte inferior (-2σ): 0.998997

Pontos após corte ±2σ: 12634

--- 2ª iteração (sigma-clipping 3σ iterativo) ---
Média clipped:  0.999997
Desvio padrão:  0.000396
Limiar +3σ:     1.001185
Limiar -3σ:     0.998809


In [10]:
# ============================================================
# PLOTAR CURVA DE LUZ EM FASE ORBITAL
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Parâmetros
P_orbital = 4.863  # período orbital em dias (AU Mic)
T0 = t.min()       # época de referência (primeiro tempo observado)

# Calcular fase para cada ponto
phase = ((t - T0) / P_orbital) % 1

# ============================================================
# Figura: Fase dobrada [0, 1]
# ============================================================

%matplotlib qt

fig, ax1 = plt.subplots(1, 1, figsize=(14, 7))

# Concatenar fase para dobrar o gráfico no intervalo [0, 1]
phase_doubled = np.concatenate([phase, phase - 1])
f_doubled = np.concatenate([f, f])
modelo_doubled = np.concatenate([modelo_manchas_suave, modelo_manchas_suave])

# Gráfico: Dados originais e modelo em fase dobrada
ax1.plot(phase_doubled, f_doubled, 'k.', ms=1.5, alpha=0.4, label='Dados Originais')
ax1.plot(phase_doubled, modelo_doubled, 'r.', ms=2, alpha=0.6, label='Modelo (manchas)')

ax1.set_xlabel('Fase Orbital', fontsize=13)
ax1.set_ylabel('Fluxo Normalizado', fontsize=13)
ax1.set_title(f'AU Mic - Curva de Luz em Fase Dobrada (P = {P_orbital} dias)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)
ax1.set_xlim(0, 1)

plt.tight_layout()
plt.show()

print(f"\nCurva de luz em fase orbital criada:")
print(f"  Período: {P_orbital} dias")
print(f"  Época de referência (T0): {T0:.5f} BTJD")
print(f"  Intervalo de fase: [{phase.min():.3f}, {phase.max():.3f}]")


Curva de luz em fase orbital criada:
  Período: 4.863 dias
  Época de referência (T0): 3883.00679 BTJD
  Intervalo de fase: [0.000, 1.000]


In [11]:
# ============================================================
# FILTRAR MODELO E REFAZER GRÁFICO DE FASE
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

# 1. Calcular média e desvio padrão do modelo de manchas
model_mean = np.mean(modelo_manchas_suave)
model_std = np.std(modelo_manchas_suave)

# 2. Definir o limiar de corte (2-sigma acima da média)
threshold = model_mean + 2 * model_std

# 3. Criar uma máscara para manter apenas os pontos ABAIXO do limiar
# Isso remove as partes mais brilhantes (e mais ativas) modeladas como manchas
mask_filtered = modelo_manchas_suave < threshold

print(f"Média do modelo: {model_mean:.4f}")
print(f"Desvio padrão do modelo: {model_std:.4f}")
print(f"Limiar de corte (média + 2σ): {threshold:.4f}")
print(f"Pontos removidos pela filtragem: {np.sum(~mask_filtered)} de {len(f)}")

# 4. Aplicar a máscara aos dados e à fase
phase_filtered = phase[mask_filtered]
f_filtered = f[mask_filtered]
model_filtered = modelo_manchas_suave[mask_filtered]

# ============================================================
# Figura: Fase dobrada com dados filtrados
# ============================================================
%matplotlib qt

fig, ax1 = plt.subplots(1, 1, figsize=(14, 7))

# Concatenar fase para dobrar o gráfico no intervalo [0, 1]
phase_doubled = np.concatenate([phase_filtered, phase_filtered - 1])
f_doubled = np.concatenate([f_filtered, f_filtered])
model_doubled = np.concatenate([model_filtered, model_filtered])

# Gráfico: Dados e modelo filtrados em fase dobrada
ax1.plot(phase_doubled, f_doubled, 'k.', ms=2, alpha=0.5, label='Dados Filtrados')
ax1.plot(phase_doubled, model_doubled, 'r.', ms=2.5, alpha=0.7, label='Modelo Filtrado')

ax1.set_xlabel('Fase Orbital', fontsize=13)
ax1.set_ylabel('Fluxo Normalizado', fontsize=13)
ax1.set_title(f'AU Mic - Curva de Luz em Fase Filtrada (P = {P_orbital} dias)', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)
ax1.set_xlim(0, 1)

plt.tight_layout()
plt.show()


Média do modelo: 1.0002
Desvio padrão do modelo: 0.0321
Limiar de corte (média + 2σ): 1.0643
Pontos removidos pela filtragem: 0 de 15111


In [12]:
# ============================================================
# PHASED LIGHT CURVE (P = 4.862 d) COM LIMPEZA DE OUTLIERS
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from astropy.stats import sigma_clip

# Periodo medido de rotacao
P_rot = 4.862  # dias
T0 = np.nanmin(t)

# 1) Remover flares/transitos da mascara ja definida no notebook
# mask_good_flares = True para pontos bons
base_mask = (
    np.isfinite(t) &
    np.isfinite(f) &
    np.isfinite(modelo_manchas_suave) &
    mask_good_flares
)

# 2) Remover variacao de manchas pelo modelo (fluxo corrigido)
flux_corr = f / modelo_manchas_suave

# 3) Cortar tudo que sai da media da curva (sigma-clipping)
flux_base = flux_corr[base_mask]
time_base = t[base_mask]

clipped = sigma_clip(flux_base, sigma=2.0, maxiters=5)
keep = ~clipped.mask

flux_clean = flux_base[keep]
time_clean = time_base[keep]

print(f'Pontos totais: {len(t)}')
print(f'Pontos apos mascara (flares/transitos): {len(flux_base)}')
print(f'Pontos apos corte em torno da media (2sigma): {len(flux_clean)}')

# 4) Curva em fase [0, 1)
phase = ((time_clean - T0) / P_rot) % 1.0

# Ordenar por fase
ord_idx = np.argsort(phase)
phase_s = phase[ord_idx]
flux_s = flux_clean[ord_idx]

# 5) Ajuste spline cubica (linha vermelha)
# s controla suavizacao; valor proporcional ao numero de pontos
s_factor = 0.5 * len(phase_s) * np.nanvar(flux_s)
spline = UnivariateSpline(phase_s, flux_s, k=3, s=s_factor)

phase_grid = np.linspace(0.0, 1.0, 1000)
flux_spline = spline(phase_grid)

%matplotlib qt
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# Pontos pretos: dados limpos em fase
ax.scatter(phase_s, flux_s, s=7, c='black', alpha=0.5, label='TESS 2018 (limpo)')

# Linha vermelha: melhor spline cubica
ax.plot(phase_grid, flux_spline, color='red', lw=2.2, label='Spline cubica (melhor ajuste)')

ax.set_xlim(0, 1)
ax.set_xlabel(f'Fase (P = {P_rot:.3f} d)', fontsize=13)
ax.set_ylabel('Fluxo normalizado corrigido', fontsize=13)
ax.set_title('Phased light curve of AU Mic', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

Pontos totais: 15111
Pontos apos mascara (flares/transitos): 13186
Pontos apos corte em torno da media (2sigma): 11070


In [13]:
# ============================================================
# NOVA CELULA: AJUSTE APENAS NO MODELO (SEM CORTAR DADOS)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from astropy.stats import sigma_clip

P_rot = 4.862
T0 = np.nanmin(t)

# 1) Dados observados: remove apenas flares/transitos da mascara
mask_data = (
    np.isfinite(t) &
    np.isfinite(f) &
    mask_good_flares
)

t_data = t[mask_data]
f_data = f[mask_data]

# 2) Modelo: aplicar corte +/-2sigma somente no modelo
mask_model = (
    np.isfinite(t) &
    np.isfinite(modelo_manchas_suave) &
    mask_good_flares
)

t_model = t[mask_model]
model_raw = modelo_manchas_suave[mask_model]

model_clip = sigma_clip(model_raw, sigma=2.0, maxiters=5)
keep_model = ~model_clip.mask

t_model_clean = t_model[keep_model]
model_clean = model_raw[keep_model]

print(f'Pontos dados (sem corte sigma): {len(f_data)}')
print(f'Pontos modelo antes do corte: {len(model_raw)}')
print(f'Pontos modelo apos corte +/-2sigma: {len(model_clean)}')

# 3) Fase dos dados e do modelo
phase_data = ((t_data - T0) / P_rot) % 1.0
phase_model = ((t_model_clean - T0) / P_rot) % 1.0

# 4) Spline cubica ajustada SOMENTE no modelo
ord_model = np.argsort(phase_model)
phase_model_s = phase_model[ord_model]
model_s = model_clean[ord_model]

# Remover fases duplicadas para evitar problemas numericos no ajuste
phase_model_u, idx_u = np.unique(phase_model_s, return_index=True)
model_u = model_s[idx_u]

# Extensao periodica para a spline seguir melhor o modelo nas bordas
phase_ext = np.concatenate([phase_model_u - 1.0, phase_model_u, phase_model_u + 1.0])
model_ext = np.concatenate([model_u, model_u, model_u])

# Spline cubica com suavizacao menor para acompanhar melhor o modelo
s_factor = 0.009 * len(phase_model_u) * np.nanvar(model_u)
spline_model = UnivariateSpline(phase_ext, model_ext, k=4, s=s_factor)

phase_grid = np.linspace(0.0, 1.0, 1200)
model_spline = spline_model(phase_grid)

%matplotlib qt
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# Pontos pretos: dados observados (sem corte sigma)
ax.scatter(phase_data, f_data, s=7, c='black', alpha=0.45, label='Dados observados (sem corte sigma)')

# Linha vermelha: modelo ajustado (corte sigma so no modelo)
ax.plot(phase_grid, model_spline, color='red', lw=2.3, label='Modelo + spline cubica (+/-2sigma no modelo)')

ax.set_xlim(0, 1)
ax.set_xlabel(f'Fase (P = {P_rot:.3f} d)', fontsize=13)
ax.set_ylabel('Fluxo normalizado', fontsize=13)
ax.set_title('AU Mic - Curva em fase com ajuste apenas no modelo', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

Pontos dados (sem corte sigma): 13186
Pontos modelo antes do corte: 13186
Pontos modelo apos corte +/-2sigma: 13186


In [14]:
# ============================================================
# NOVA CELULA: REAJUSTE DO MODELO EM FASE + AJUSTE GAUSSIANO
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from scipy.ndimage import gaussian_filter1d
from astropy.stats import sigma_clip

P_rot = 4.862
T0 = np.nanmin(t)

# 1) Dados observados: remover apenas flares/transitos
mask_data = (
    np.isfinite(t) &
    np.isfinite(f) &
    mask_good_flares
)

t_data = t[mask_data]
f_data = f[mask_data]

# 2) Modelo: corte apenas no modelo com +/-2sigma
mask_model = (
    np.isfinite(t) &
    np.isfinite(modelo_manchas_suave) &
    mask_good_flares
)

t_model = t[mask_model]
model_raw = modelo_manchas_suave[mask_model]

model_clip = sigma_clip(model_raw, sigma=2.0, maxiters=5)
keep_model = ~model_clip.mask

t_model_clean = t_model[keep_model]
model_clean = model_raw[keep_model]

print(f'Pontos dados (sem corte sigma): {len(f_data)}')
print(f'Pontos modelo antes do corte: {len(model_raw)}')
print(f'Pontos modelo apos corte +/-2sigma: {len(model_clean)}')

# 3) Colocar em fase
phase_data = ((t_data - T0) / P_rot) % 1.0
phase_model = ((t_model_clean - T0) / P_rot) % 1.0

# 4) Binning em fase para estabilizar o ajuste do modelo
nbins = 120
bins = np.linspace(0.0, 1.0, nbins + 1)
centers = 0.5 * (bins[:-1] + bins[1:])

phase_bin = []
model_bin = []

for i in range(nbins):
    in_bin = (phase_model >= bins[i]) & (phase_model < bins[i + 1])
    if np.sum(in_bin) >= 5:
        phase_bin.append(centers[i])
        model_bin.append(np.nanmedian(model_clean[in_bin]))

phase_bin = np.array(phase_bin)
model_bin = np.array(model_bin)

# 5) Extensao periodica para bordas suaves
phase_ext = np.concatenate([phase_bin - 1.0, phase_bin, phase_bin + 1.0])
model_ext = np.concatenate([model_bin, model_bin, model_bin])
ord = np.argsort(phase_ext)
phase_ext = phase_ext[ord]
model_ext = model_ext[ord]

# 6) Ajuste spline cubica no modelo binned
s_factor = 0.002 * len(phase_bin) * np.nanvar(model_bin)
spline_model = UnivariateSpline(phase_ext, model_ext, k=3, s=s_factor)

phase_grid = np.linspace(0.0, 1.0, 1200)
model_spline = spline_model(phase_grid)

# 7) Ajuste gaussiano por suavizacao gaussiana no modelo binned
model_gauss_base = gaussian_filter1d(model_bin, sigma=3, mode='wrap')
phase_gauss_ext = np.concatenate([phase_bin - 1.0, phase_bin, phase_bin + 1.0])
model_gauss_ext = np.concatenate([model_gauss_base, model_gauss_base, model_gauss_base])
ord_g = np.argsort(phase_gauss_ext)
phase_gauss_ext = phase_gauss_ext[ord_g]
model_gauss_ext = model_gauss_ext[ord_g]
model_gauss = np.interp(phase_grid, phase_gauss_ext, model_gauss_ext)

%matplotlib qt
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# Dados observados
ax.scatter(phase_data, f_data, s=7, c='black', alpha=0.35, label='Dados observados')

# Modelo reajustado por spline
ax.plot(phase_grid, model_spline, color='red', lw=2.3, label='Modelo reajustado + spline cubica')

# Modelo suavizado gaussianamente
ax.plot(phase_grid, model_gauss, color='royalblue', lw=2.0, ls='--', label='Modelo com ajuste gaussiano')

# Pontos binned do modelo para referencia
ax.plot(phase_bin, model_bin, 'o', ms=3, color='darkorange', alpha=0.8, label='Modelo binned')

ax.set_xlim(0, 1)
ax.set_xlabel(f'Fase (P = {P_rot:.3f} d)', fontsize=13)
ax.set_ylabel('Fluxo normalizado', fontsize=13)
ax.set_title('AU Mic - Reajuste do modelo em fase', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

Pontos dados (sem corte sigma): 13186
Pontos modelo antes do corte: 13186
Pontos modelo apos corte +/-2sigma: 13186


In [15]:
# ============================================================
# NOVA CELULA: MODELO ADAPTADO NA CURVA TODA (SEM COLOCAR EM FASE)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from scipy.ndimage import gaussian_filter1d

# 1) Pontos bons: remover apenas flares/transitos
mask_time = (
    np.isfinite(t) &
    np.isfinite(f) &
    np.isfinite(modelo_manchas_suave) &
    mask_good_flares
)

t_good = t[mask_time]
f_good = f[mask_time]
model_good = modelo_manchas_suave[mask_time]

# 2) Ordenar no tempo
ord_t = np.argsort(t_good)
t_ord = t_good[ord_t]
f_ord = f_good[ord_t]
model_ord = model_good[ord_t]

# 3) Binning temporal inspirado na logica da celula 4, mas no dominio do tempo
nbins_time = 180
edges_t = np.linspace(t_ord.min(), t_ord.max(), nbins_time + 1)
centers_t = 0.5 * (edges_t[:-1] + edges_t[1:])

bin_t = []
bin_model = []

for i in range(nbins_time):
    in_bin = (t_ord >= edges_t[i]) & (t_ord < edges_t[i + 1])
    if np.sum(in_bin) >= 5:
        bin_t.append(centers_t[i])
        bin_model.append(np.nanmedian(model_ord[in_bin]))

bin_t = np.array(bin_t)
bin_model = np.array(bin_model)

# 4) Reajuste do modelo no tempo
s_factor_time = 0.001 * len(bin_t) * np.nanvar(bin_model)
spline_time = UnivariateSpline(bin_t, bin_model, k=4, s=s_factor_time)
model_spline_time = spline_time(t_ord)

# 5) Ajuste gaussiano no tempo para comparar
model_gauss_bin = gaussian_filter1d(bin_model, sigma=3)
model_gauss_time = np.interp(t_ord, bin_t, model_gauss_bin)

# 6) Residuais usando os modelos reajustados
residual_spline_time = f_ord / model_spline_time
residual_gauss_time = f_ord / model_gauss_time

%matplotlib qt
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 10), sharex=True)

# Painel superior: curva total + modelos
ax1.plot(t_ord, f_ord, 'k.', ms=2, alpha=0.35, label='Dados observados')
ax1.plot(t_ord, model_spline_time, color='red', lw=2.2, label='Modelo adaptado (spline)')
ax1.plot(t_ord, model_gauss_time, color='royalblue', lw=2.0, ls='--', label='Modelo adaptado (gaussiano)')
ax1.plot(bin_t, bin_model, 'o', ms=3, color='darkorange', alpha=0.8, label='Modelo binned')
ax1.set_ylabel('Fluxo normalizado', fontsize=13)
ax1.set_title('AU Mic - Modelo adaptado na curva inteira', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.legend(fontsize=10)

# Painel inferior: residual com o modelo spline
ax2.plot(t_ord, residual_spline_time, 'b.', ms=2, alpha=0.35, label='Residual / modelo spline')
ax2.axhline(1.0, color='gray', ls='--', lw=1.2, alpha=0.7)
ax2.set_xlabel('Tempo [BTJD]', fontsize=13)
ax2.set_ylabel('Fluxo residual', fontsize=13)
ax2.grid(alpha=0.3)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f'Pontos usados no ajuste temporal: {len(t_ord)}')
print(f'Bins temporais usados: {len(bin_t)}')

Pontos usados no ajuste temporal: 13186
Bins temporais usados: 159


In [86]:
# ============================================================
# CELULA FINAL: MODELO FINAL UNIDO (SPLINE + BINNED) + RESIDUO
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline

# -----------------------------
# 1) Selecionar pontos bons
# -----------------------------
mask_final = (
    np.isfinite(t) &
    np.isfinite(f) &
    np.isfinite(modelo_manchas_suave) &
    mask_good_flares
)

t_final = t[mask_final]
f_final = f[mask_final]
modelo_base = modelo_manchas_suave[mask_final]

# -----------------------------
# 2) Ordenar no tempo
# -----------------------------
ord_final = np.argsort(t_final)

t_final = t_final[ord_final]
f_final = f_final[ord_final]
modelo_base = modelo_base[ord_final]

# -----------------------------
# 3) Binning temporal do modelo base
# -----------------------------
nbins_final = 380

edges_final = np.linspace(t_final.min(), t_final.max(), nbins_final + 1)
centers_final = 0.5 * (edges_final[:-1] + edges_final[1:])

bin_t_final = []
bin_model_final = []

for i in range(nbins_final):
    in_bin = (t_final >= edges_final[i]) & (t_final < edges_final[i + 1])

    if np.sum(in_bin) >= 5:
        bin_t_final.append(centers_final[i])
        bin_model_final.append(np.nanmedian(modelo_base[in_bin]))

bin_t_final = np.array(bin_t_final)
bin_model_final = np.array(bin_model_final)

# -----------------------------
# 4) Modelo spline no tempo
# -----------------------------
s_factor_final = 0.001 * len(bin_t_final) * np.nanvar(bin_model_final)

spline_final = UnivariateSpline(
    bin_t_final,
    bin_model_final,
    k=4,
    s=s_factor_final
)

modelo_final = spline_final(t_final)

# -----------------------------
# 5) Colocar o modelo binned na mesma grade de tempo
# -----------------------------
bin_model_em_t = np.interp(
    t_final,
    bin_t_final,
    bin_model_final
)

# -----------------------------
# 6) Unir modelo spline + modelo binned
# -----------------------------
# peso_bin maior = acompanha mais os bins
# peso_bin menor = modelo mais suave
peso_bin = 0.65
peso_spline = 1.0 - peso_bin

modelo_final_unido = (
    peso_spline * modelo_final +
    peso_bin * bin_model_em_t
)

# -----------------------------
# 7) Residuo final
# -----------------------------
residuo_final = f_final / modelo_final_unido

# ============================================================
# GRAFICO
# ============================================================

%matplotlib qt

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(13, 10),
    sharex=True,
    gridspec_kw={'height_ratios': [2.3, 1]}
)

# -----------------------------
# Painel superior: dados + modelo unido
# -----------------------------
ax1.plot(
    t_final,
    f_final,
    'k.-',
    ms=1.5,
    lw=0.5,
    alpha=0.45,
    label='Dados Originais',
    zorder=1
)

ax1.plot(
    t_final,
    modelo_final_unido,
    'r-',
    lw=2.2,
    label='Modelo final unido',
    zorder=3
)

ax1.plot(
    bin_t_final,
    bin_model_final,
    'o',
    ms=3,
    color='darkorange',
    alpha=0.75,
    label='Modelo binned',
    zorder=4
)

ax1.set_ylabel('Fluxo Normalizado', fontsize=13)
ax1.set_title('AU Mic - Modelo final unido: spline + binned', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(alpha=0.3)

# -----------------------------
# Painel inferior: residual final
# -----------------------------
ax2.plot(
    t_final,
    residuo_final,
    'b.-',
    ms=1.5,
    lw=0.5,
    alpha=0.6,
    label='Residual final',
    zorder=1
)

ax2.axhline(
    1.0,
    color='gray',
    ls='--',
    lw=1.2,
    alpha=0.7
)

ax2.set_xlabel('Tempo [BTJD]', fontsize=13)
ax2.set_ylabel('Fluxo Residual', fontsize=13)
ax2.set_title('Residual do modelo final unido', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================
# ESTATISTICAS
# ============================================================

print("\n" + "="*60)
print("MODELO FINAL UNIDO: SPLINE + BINNED")
print("="*60)
print(f"Pontos usados no ajuste final: {len(t_final)}")
print(f"Bins usados no modelo final:    {len(bin_t_final)}")
print(f"Peso spline:                   {peso_spline:.2f}")
print(f"Peso binned:                   {peso_bin:.2f}")
print("-"*60)
print(f"Residual final:")
print(f"  Média:           {np.nanmean(residuo_final):.6f}")
print(f"  Desvio padrão:   {np.nanstd(residuo_final):.6f}")
print(f"  Mediana:         {np.nanmedian(residuo_final):.6f}")
print(f"  Min/Max:         {np.nanmin(residuo_final):.6f} / {np.nanmax(residuo_final):.6f}")


MODELO FINAL UNIDO: SPLINE + BINNED
Pontos usados no ajuste final: 13187
Bins usados no modelo final:    319
Peso spline:                   0.35
Peso binned:                   0.65
------------------------------------------------------------
Residual final:
  Média:           1.000019
  Desvio padrão:   0.000634
  Mediana:         0.999995
  Min/Max:         0.996989 / 1.006179


In [85]:
# ============================================================
# CELULA FINAL: MODELO FINAL (SPLINE NO TEMPO) + RESIDUO FINAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline

# 1) Selecionar pontos bons
mask_final = (
    np.isfinite(t) &
    np.isfinite(f) &
    np.isfinite(modelo_manchas_suave) &
    mask_good_flares
)

t_final = t[mask_final]
f_final = f[mask_final]
modelo_base = modelo_manchas_suave[mask_final]

# 2) Ordenar no tempo
ord_final = np.argsort(t_final)
t_final = t_final[ord_final]
f_final = f_final[ord_final]
modelo_base = modelo_base[ord_final]

# 3) Binning temporal do modelo base
nbins_final = 380
edges_final = np.linspace(t_final.min(), t_final.max(), nbins_final + 1)
centers_final = 0.5 * (edges_final[:-1] + edges_final[1:])

bin_t_final = []
bin_model_final = []

for i in range(nbins_final):
    in_bin = (t_final >= edges_final[i]) & (t_final < edges_final[i + 1])
    if np.sum(in_bin) >= 5:
        bin_t_final.append(centers_final[i])
        bin_model_final.append(np.nanmedian(modelo_base[in_bin]))

bin_t_final = np.array(bin_t_final)
bin_model_final = np.array(bin_model_final)

# 4) Modelo final: spline no tempo
s_factor_final = 0.001 * len(bin_t_final) * np.nanvar(bin_model_final)
spline_final = UnivariateSpline(bin_t_final, bin_model_final, k=4, s=s_factor_final)
modelo_final = spline_final(t_final)

# 5) Residuo final
residuo_final = f_final / modelo_final

%matplotlib qt
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 10), sharex=True)

# Painel superior: dados + modelo final
ax1.plot(t_final, f_final, 'k.-', ms=1.5, lw=0.5, alpha=0.5, label='Dados Originais', zorder=1)
ax1.plot(t_final, modelo_final, 'r-', lw=2.2, label='Modelo final (spline)', zorder=3)
ax1.plot(bin_t_final, bin_model_final, 'o', ms=3, color='darkorange', alpha=0.8, label='Modelo binned', zorder=4)
ax1.set_ylabel('Fluxo Normalizado', fontsize=13)
ax1.set_title('AU Mic - Ajuste final do modelo na curva inteira', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(alpha=0.3)

# Painel inferior: residual final
ax2.plot(t_final, residuo_final, 'b.-', ms=1.5, lw=0.5, alpha=0.6, label='Residual final', zorder=1)
ax2.axhline(1.0, color='gray', ls='--', lw=1.2, alpha=0.7)
ax2.set_xlabel('Tempo [BTJD]', fontsize=13)
ax2.set_ylabel('Fluxo Residual', fontsize=13)
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Pontos usados no ajuste final: {len(t_final)}')
print(f'Bins usados no modelo final: {len(bin_t_final)}')

Pontos usados no ajuste final: 13187
Bins usados no modelo final: 319


In [22]:
# ============================================================
# COMPARACAO: MODELO ORIGINAL vs MODELO ADAPTADO (SPLINE)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

# Garantir que temos os modelos e dados necessarios
mask_comp = (
    np.isfinite(t) &
    np.isfinite(f) &
    np.isfinite(modelo_manchas_suave) &
    mask_good_flares
)

t_comp = t[mask_comp]
f_comp = f[mask_comp]
modelo_orig = modelo_manchas_suave[mask_comp]

# Ordenar no tempo para plotar corretamente
ord_comp = np.argsort(t_comp)
t_comp = t_comp[ord_comp]
f_comp = f_comp[ord_comp]
modelo_orig = modelo_orig[ord_comp]

# Interpolacao do modelo adaptado para os mesmos tempos
from scipy.interpolate import UnivariateSpline
nbins_comp = 380
edges_comp = np.linspace(t_comp.min(), t_comp.max(), nbins_comp + 1)
centers_comp = 0.5 * (edges_comp[:-1] + edges_comp[1:])

bin_t_comp = []
bin_model_comp = []

for i in range(nbins_comp):
    in_bin = (t_comp >= edges_comp[i]) & (t_comp < edges_comp[i + 1])
    if np.sum(in_bin) >= 5:
        bin_t_comp.append(centers_comp[i])
        bin_model_comp.append(np.nanmedian(modelo_orig[in_bin]))

bin_t_comp = np.array(bin_t_comp)
bin_model_comp = np.array(bin_model_comp)

# Modelo adaptado por spline
s_factor_comp = 0.001 * len(bin_t_comp) * np.nanvar(bin_model_comp)
spline_comp = UnivariateSpline(bin_t_comp, bin_model_comp, k=4, s=s_factor_comp)
modelo_adaptado = spline_comp(t_comp)

# Residuais
residual_original = f_comp / modelo_orig
residual_adaptado = f_comp / modelo_adaptado

%matplotlib qt
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10), sharex='col')

# ====== Painel 1: Dados + Modelo Original ======
ax1.plot(t_comp, f_comp, 'k.', ms=1.5, alpha=0.3, label='Dados Originais')
ax1.plot(t_comp, modelo_orig, 'r-', lw=2.2, label='Modelo Original', zorder=3)
ax1.set_ylabel('Fluxo Normalizado', fontsize=12)
ax1.set_title('Modelo Original (manchas)', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(alpha=0.3)

# ====== Painel 2: Dados + Modelo Adaptado ======
ax2.plot(t_comp, f_comp, 'k.', ms=1.5, alpha=0.3, label='Dados Originais')
ax2.plot(t_comp, modelo_adaptado, 'b-', lw=2.2, label='Modelo Adaptado (spline)', zorder=3)
ax2.plot(bin_t_comp, bin_model_comp, 'o', ms=2.5, color='darkorange', alpha=0.7, label='Binned', zorder=4)
ax2.set_ylabel('Fluxo Normalizado', fontsize=12)
ax2.set_title('Modelo Adaptado (spline)', fontsize=13, fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(alpha=0.3)

# ====== Painel 3: Residual Original ======
ax3.plot(t_comp, residual_original, 'r.', ms=1.5, alpha=0.5, label='Residual Original')
ax3.axhline(1.0, color='gray', ls='--', lw=1.2, alpha=0.7)
ax3.set_xlabel('Tempo [BTJD]', fontsize=12)
ax3.set_ylabel('Fluxo Residual', fontsize=12)
ax3.set_title('Residual: Dados / Modelo Original', fontsize=13, fontweight='bold')
ax3.legend(loc='upper right', fontsize=9)
ax3.grid(alpha=0.3)

# ====== Painel 4: Residual Adaptado ======
ax4.plot(t_comp, residual_adaptado, 'b.', ms=1.5, alpha=0.5, label='Residual Adaptado')
ax4.axhline(1.0, color='gray', ls='--', lw=1.2, alpha=0.7)
ax4.set_xlabel('Tempo [BTJD]', fontsize=12)
ax4.set_ylabel('Fluxo Residual', fontsize=12)
ax4.set_title('Residual: Dados / Modelo Adaptado', fontsize=13, fontweight='bold')
ax4.legend(loc='upper right', fontsize=9)
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Estatisticas dos residuais
print("\n" + "="*60)
print("COMPARACAO DE RESIDUAIS")
print("="*60)
print(f"Residual Original:")
print(f"  Média:           {np.mean(residual_original):.6f}")
print(f"  Desvio padrão:   {np.std(residual_original):.6f}")
print(f"  Min/Max:         {np.min(residual_original):.6f} / {np.max(residual_original):.6f}")
print(f"\nResidual Adaptado:")
print(f"  Média:           {np.mean(residual_adaptado):.6f}")
print(f"  Desvio padrão:   {np.std(residual_adaptado):.6f}")
print(f"  Min/Max:         {np.min(residual_adaptado):.6f} / {np.max(residual_adaptado):.6f}")


COMPARACAO DE RESIDUAIS
Residual Original:
  Média:           1.000026
  Desvio padrão:   0.000514
  Min/Max:         0.997640 / 1.006015

Residual Adaptado:
  Média:           1.000025
  Desvio padrão:   0.001055
  Min/Max:         0.994791 / 1.006279


In [21]:
# ============================================================
# MODELO FINAL: 1 RESIDUO UNICO (dados originais com flares/transitos)
# Etapa 1: divide pela componente de manchas (modelo original)
# Etapa 2: remove tendencia residual com spline suave
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline

# 1) Dados originais validos (MANTENDO flares/transitos)
mask_adapt = (
    np.isfinite(t) &
    np.isfinite(f) &
    np.isfinite(modelo_manchas_suave)
)

t_adapt = t[mask_adapt]
f_adapt = f[mask_adapt]
modelo_orig_adapt = modelo_manchas_suave[mask_adapt]

# Ordenar no tempo
ord_a = np.argsort(t_adapt)
t_adapt = t_adapt[ord_a]
f_adapt = f_adapt[ord_a]
modelo_orig_adapt = modelo_orig_adapt[ord_a]

# 2) PRIMEIRA divisao: remove variacao principal de manchas
residual_1 = f_adapt / modelo_orig_adapt

# 3) Ajustar spline no residual_1 para capturar tendencia lenta remanescente
nbins_r = 420
edges_r = np.linspace(t_adapt.min(), t_adapt.max(), nbins_r + 1)
centers_r = 0.5 * (edges_r[:-1] + edges_r[1:])

bin_t_r = []
bin_res_r = []

for i in range(nbins_r):
    in_bin = (t_adapt >= edges_r[i]) & (t_adapt < edges_r[i + 1])
    if np.sum(in_bin) >= 5:
        bin_t_r.append(centers_r[i])
        bin_res_r.append(np.nanmedian(residual_1[in_bin]))

bin_t_r = np.array(bin_t_r)
bin_res_r = np.array(bin_res_r)

# Spline suave no residual de primeira etapa
s_factor_r = 0.0025 * len(bin_t_r) * np.nanvar(bin_res_r)
spline_r = UnivariateSpline(bin_t_r, bin_res_r, k=3, s=s_factor_r)
trend_residual = spline_r(t_adapt)

# 4) SEGUNDA divisao: residual final unico e mais suave
residual_final = residual_1 / trend_residual

# Curva corrigida final (opcional): deve ficar centrada e suave
fluxo_corrigido = f_adapt / (modelo_orig_adapt * trend_residual)

# Estatisticas de suavidade
std_r1 = np.nanstd(residual_1)
std_rf = np.nanstd(residual_final)

print('\n' + '=' * 70)
print('RESIDUO UNICO EM DUAS ETAPAS')
print('=' * 70)
print(f'Pontos usados: {len(t_adapt)} (com flares/transitos)')
print(f'Desvio padrao residual_1 (f/modelo_original): {std_r1:.6f}')
print(f'Desvio padrao residual_final (apos spline):   {std_rf:.6f}')
print(f'Ganho de suavizacao: {(1 - std_rf/std_r1)*100:.2f}%')

# 5) Visualizacao
%matplotlib qt
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Painel 1: dados e modelo original
ax1.plot(t_adapt, f_adapt, 'k.', ms=1.2, alpha=0.30, label='Dados originais (com flares/transitos)')
ax1.plot(t_adapt, modelo_orig_adapt, color='red', lw=2.0, alpha=0.85, label='Modelo original (manchas)')
ax1.set_ylabel('Fluxo')
ax1.set_title('Etapa 1: Dados / Modelo Original', fontweight='bold')
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(alpha=0.3)

# Painel 2: residual_1 + tendencia spline
ax2.plot(t_adapt, residual_1, '.', color='royalblue', ms=1.0, alpha=0.45, label='Residual 1 = f / modelo_original')
ax2.plot(t_adapt, trend_residual, color='darkorange', lw=2.0, alpha=0.95, label='Spline no residual_1')
ax2.axhline(1.0, color='gray', ls='--', lw=1.0, alpha=0.7)
ax2.set_ylabel('Residual 1')
ax2.set_title('Etapa 2: Ajuste spline no residual', fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)
ax2.grid(alpha=0.3)

# Painel 3: residual final unico
ax3.plot(t_adapt, residual_final, 'g.', ms=1.1, alpha=0.55, label='Residual final unico')
ax3.axhline(1.0, color='gray', ls='--', lw=1.0, alpha=0.8)
ax3.set_xlabel('Tempo [BTJD]')
ax3.set_ylabel('Residual final')
ax3.set_title('Resultado final (mais suave)', fontweight='bold')
ax3.legend(loc='upper right', fontsize=9)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()


RESIDUO UNICO EM DUAS ETAPAS
Pontos usados: 15111 (com flares/transitos)
Desvio padrao residual_1 (f/modelo_original): 0.002086
Desvio padrao residual_final (apos spline):   0.001538
Ganho de suavizacao: 26.27%


In [32]:
# ============================================================
# FUSAO: MODELO ORIGINAL + ADAPTADO -> MODELO COMBINADO UNICO
# Objetivo: maximizar aderencia aos dados, minimizar residuo
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline

# ----------------------------------------------------------
# 1. MASCARA E ORDENACAO
# ----------------------------------------------------------
mask_sobre = (
    np.isfinite(t) &
    np.isfinite(f) &
    np.isfinite(modelo_manchas_suave) &
    mask_good_flares
)

t_s     = t[mask_sobre]
f_s     = f[mask_sobre]
mod_s   = modelo_manchas_suave[mask_sobre]

ord_s   = np.argsort(t_s)
t_s     = t_s[ord_s]
f_s     = f_s[ord_s]
mod_s   = mod_s[ord_s]

# ----------------------------------------------------------
# 2. SPLINE DO MODELO ORIGINAL (captura tendencia global)
# ----------------------------------------------------------
nbins_orig = 380
edges_orig  = np.linspace(t_s.min(), t_s.max(), nbins_orig + 1)
centers_orig = 0.5 * (edges_orig[:-1] + edges_orig[1:])

bin_t_orig, bin_m_orig = [], []
for i in range(nbins_orig):
    mask_bin = (t_s >= edges_orig[i]) & (t_s < edges_orig[i + 1])
    if np.sum(mask_bin) >= 5:
        bin_t_orig.append(centers_orig[i])
        bin_m_orig.append(np.nanmedian(mod_s[mask_bin]))

bin_t_orig = np.array(bin_t_orig)
bin_m_orig = np.array(bin_m_orig)

s_orig       = 0.001 * len(bin_t_orig) * np.nanvar(bin_m_orig)
spline_orig  = UnivariateSpline(bin_t_orig, bin_m_orig, k=4, s=s_orig)
mod_spline_s = spline_orig(t_s)          # modelo original suavizado

# ----------------------------------------------------------
# 3. RESIDUO PARCIAL E CORRECAO LOCAL (spline dos dados / mod_orig)
# ----------------------------------------------------------
residuo_parcial = f_s / mod_spline_s      # desvio local dos dados em relacao ao modelo

# Binar o residuo parcial (alta resolucao para capturar variacao local)
nbins_corr  = 600                         # mais bins -> mais aderencia local
edges_corr  = np.linspace(t_s.min(), t_s.max(), nbins_corr + 1)
centers_corr = 0.5 * (edges_corr[:-1] + edges_corr[1:])

bin_t_corr, bin_r_corr = [], []
for i in range(nbins_corr):
    mask_bin = (t_s >= edges_corr[i]) & (t_s < edges_corr[i + 1])
    if np.sum(mask_bin) >= 3:             # menos pontos por bin -> mais resolucao
        bin_t_corr.append(centers_corr[i])
        bin_r_corr.append(np.nanmedian(residuo_parcial[mask_bin]))

bin_t_corr = np.array(bin_t_corr)
bin_r_corr = np.array(bin_r_corr)

# Spline da correcao local (suavizacao minima para nao overfitar flares)
s_corr      = 0.0005 * len(bin_t_corr) * np.nanvar(bin_r_corr)
spline_corr = UnivariateSpline(bin_t_corr, bin_r_corr, k=4, s=s_corr)
correcao    = spline_corr(t_s)

# ----------------------------------------------------------
# 4. MODELO COMBINADO = modelo_original_spline * correcao_local
# ----------------------------------------------------------
modelo_combinado = mod_spline_s * correcao

# ----------------------------------------------------------
# 5. RESIDUO FINAL
# ----------------------------------------------------------
residuo_final = f_s / modelo_combinado

# ----------------------------------------------------------
# 6. GRAFICO 1 — MODELO ORIGINAL (manchas) sobre os dados
# ----------------------------------------------------------
%matplotlib qt

fig1, ax1 = plt.subplots(figsize=(15, 5))
ax1.plot(t_s, f_s, 'k.', ms=1.2, alpha=0.25, label='Dados', zorder=1)
ax1.plot(t_s, mod_spline_s, color='tomato', lw=1.8,
         label='Modelo Original — spline das manchas (tendência global)', zorder=3, alpha=0.9)
ax1.set_xlabel('Tempo [BTJD]', fontsize=13)
ax1.set_ylabel('Fluxo Normalizado', fontsize=13)
ax1.set_title('AU Mic — Componente 1: Modelo Original (Manchas / Spline Global)',
              fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------
# 7. GRAFICO 2 — CORRECAO LOCAL (residuo parcial e seu spline)
# ----------------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(15, 5))
ax2.plot(t_s, residuo_parcial, 'k.', ms=1.2, alpha=0.20,
         label='Resíduo parcial (Dados / Modelo Original)', zorder=1)
ax2.plot(bin_t_corr, bin_r_corr, 'o', ms=2.0, color='darkorange',
         alpha=0.6, label='Binned do resíduo parcial', zorder=2)
ax2.plot(t_s, correcao, color='royalblue', lw=1.8,
         label='Correção Local — spline do resíduo parcial', zorder=3, alpha=0.9)
ax2.axhline(1.0, color='gray', ls='--', lw=1.0, alpha=0.6, zorder=0)
ax2.set_xlabel('Tempo [BTJD]', fontsize=13)
ax2.set_ylabel('Fator de Correção', fontsize=13)
ax2.set_title('AU Mic — Componente 2: Correção Local (Spline do Resíduo Parcial)',
              fontsize=14, fontweight='bold')
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# ----------------------------------------------------------
# 8. GRAFICO 3 — MODELO COMBINADO e RESIDUO FINAL
# ----------------------------------------------------------
fig3, (ax3, ax4) = plt.subplots(2, 1, figsize=(15, 9), sharex=True)

ax3.plot(t_s, f_s, 'k.', ms=1.2, alpha=0.25, label='Dados', zorder=1)
ax3.plot(t_s, modelo_combinado, color='mediumseagreen', lw=1.8,
         label='Modelo Combinado (Original × Correção Local)', zorder=3, alpha=0.9)
ax3.set_ylabel('Fluxo Normalizado', fontsize=13)
ax3.set_title('AU Mic — Modelo Combinado e Resíduo Final',
              fontsize=14, fontweight='bold')
ax3.legend(loc='upper right', fontsize=10)
ax3.grid(alpha=0.25)

ax4.plot(t_s, residuo_final, '.', ms=1.2, alpha=0.35,
         color='mediumseagreen', label='Resíduo Final (Dados / Modelo Combinado)', zorder=1)
ax4.axhline(1.0, color='crimson', ls='--', lw=1.3, alpha=0.8, zorder=2, label='Nível 1.0')
ax4.set_xlabel('Tempo [BTJD]', fontsize=13)
ax4.set_ylabel('Fluxo Residual', fontsize=13)
ax4.legend(loc='upper right', fontsize=10)
ax4.grid(alpha=0.25)

plt.tight_layout()
plt.show()

# ----------------------------------------------------------
# 8. ESTATISTICAS
# ----------------------------------------------------------
print("\n" + "="*60)
print("ESTATÍSTICAS — MODELO COMBINADO")
print("="*60)
print(f"  Média do resíduo:        {np.mean(residuo_final):.6f}")
print(f"  Desvio padrão (sigma):   {np.std(residuo_final):.6f}")
print(f"  Mediana:                 {np.median(residuo_final):.6f}")
print(f"  Min / Max:               {np.min(residuo_final):.6f}  /  {np.max(residuo_final):.6f}")
print(f"\n  Pontos usados:           {len(t_s)}")
print(f"  Bins (correção local):   {len(bin_t_corr)}")
print(f"  Fator s (correção):      {s_corr:.6e}")
print("="*60)

# ----------------------------------------------------------
# NOTA DE AJUSTE FINO
# ----------------------------------------------------------
# Para controlar o quanto o modelo "adere" aos dados vs suaviza:
#   - Diminuir nbins_corr  -> modelo mais suave (menos aderente)
#   - Aumentar  nbins_corr -> modelo mais aderente (risco de seguir flares)
#   - Diminuir  s_corr     -> spline local mais rígida (mais aderente)
#   - Aumentar  s_corr     -> spline local mais suave
# Recomendado: manter nbins_corr entre 400-700 e s_corr entre 0.0003 e 0.001



ESTATÍSTICAS — MODELO COMBINADO
  Média do resíduo:        1.000019
  Desvio padrão (sigma):   0.000468
  Mediana:                 0.999996
  Min / Max:               0.996632  /  1.003991

  Pontos usados:           13186
  Bins (correção local):   489
  Fator s (correção):      2.536121e-07


In [139]:
from astropy.stats import sigma_clip
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from lightkurve import LightCurve

# ============================================================
# PASSO 3: AJUSTE POLINOMIAL + FLATTEN LOCAL + SPLINE LOCAL
# ============================================================

print("\n" + "="*60)
print("AJUSTE: Polinomial com Segmentos + Spline em Regiões")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# ============================================================
# REGIÕES MANUAIS (poly / flatten)
# ============================================================

ajustes_manuais = [
    [3883.0050, 3883.4554, "poly", 4, 2.5],
    [3883.1409, 3883.7849, "poly", 4, 2.5],
    [3883.8264, 3884.0955, "poly", 4, 2.5],
    [3884.2828, 3885.3000, "poly", 4, 2.5],
    [3885.2839, 3885.4326, "poly", 4, 2.5],
    [3885.6205, 3885.7794, "poly", 2, 2.5],
    [3892.6280, 3892.7040, "poly", 1, 2.5],
    #[3899.86607, 3900.8691, "poly", 6, 2.5],
    #[3900.8273, 3900.8691, "flatten", None, None],
]

# ============================================================
# REGIÕES SPLINE  <-- ADICIONE / EDITE AQUI
# Formato: [t_inicio, t_fim, nbins, grau_k, fator_s]
# ============================================================

regioes_spline = [
    # [t_ini,      t_fim,   nbins, k, fator_s, sigma_clip_int]
    #[3899.86607, 3900.8691,  190,  4, 0.0001,  1.5],  # região com flares
    #[3900.8273,  3901.44,     60,  4, 0.001,   1.5],  # região mais tranquila
    # [3905.0,   3907.00,     60,  3, 0.01,    2.2],  # descomente para mais
     [3899.86607, 3900.826,   60,  3,  0.01,   1.5],  # mais suave, ignora flares
    #[3899.501, 3900.826,   60,  7,  0.01,   1.5],  # mais suave, ignora flares

    #[3900.8273,  3901.44,     30,  3,  0.01,   1.5],
]
 

# ============================================================
# REGIÕES FLATTEN LOCAL  <-- ADICIONE / EDITE AQUI
# Formato: [t_inicio, t_fim, window_length, polyorder, sigma, break_tolerance, niters]
# Exemplo: [3900.8273, 3900.9512, 320, 3, 2.5, 10, 4]
# ============================================================

regioes_flatten = [
    #[3900.5909, 3900.9512, 280, 1, 1.5, 1, 4],
    # [3887.1500, 3888.4400, 280, 2, 2.2, 8, 5],
]

# ============================================================
# FLATTEN GLOBAL (fallback para uso em ajustes_manuais do tipo flatten)
# ============================================================

flcd, trend = lc2_m.flatten(
    window_length=320,
    polyorder=3,
    return_trend=True,
    break_tolerance=10,
    niters=4,
    sigma=2.5,
    mask=mask_good_flares
)

# ============================================================
# AJUSTE AUTOMÁTICO
# ============================================================

N_SEGMENTOS_AUTO = 40
GRAU_AUTO        = 4
SIGMA_AUTO       = 3.0

# ============================================================
# FUNÇÕES
# ============================================================

def fitting_segment(t_seg, f_seg, mask_seg, deg, sigma):
    if len(t_seg) < deg + 2:
        return np.full_like(f_seg, np.nanmedian(f_seg))
    t_mid = np.median(t_seg)
    t_s   = t_seg - t_mid
    good  = mask_seg.copy()
    modelo = np.full_like(f_seg, np.nan)
    for _ in range(5):
        if np.sum(good) < deg + 2:
            break
        coef = np.polyfit(t_s[good], f_seg[good], deg=deg)
        modelo_good = np.polyval(coef, t_s[good])
        modelo = np.interp(t_s, t_s[good], modelo_good)
        resid = f_seg - modelo
        clipped = sigma_clip(resid[good], sigma=sigma, maxiters=1)
        if clipped.mask is np.ma.nomask:
            break
        good[np.where(good)[0]] = ~clipped.mask
    if np.all(np.isnan(modelo)):
        modelo = np.full_like(f_seg, np.nanmedian(f_seg))
    return modelo


def fitting_spline(t_seg, f_seg, mask_seg, nbins, k, fator_s):
    t_ok = t_seg[mask_seg]
    f_ok = f_seg[mask_seg]

    if len(t_ok) < k + 2:
        print("    [spline] pontos insuficientes, usando mediana.")
        return np.full_like(f_seg, np.nanmedian(f_seg))

    edges = np.linspace(t_ok.min(), t_ok.max(), nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    bin_t, bin_f = [], []
    for i in range(nbins):
        in_bin = (t_ok >= edges[i]) & (t_ok < edges[i + 1])
        if np.sum(in_bin) >= 1:
            bin_t.append(centers[i])
            bin_f.append(np.nanmedian(f_ok[in_bin]))

    bin_t = np.array(bin_t)
    bin_f = np.array(bin_f)

    if len(bin_t) < k + 2:
        print("    [spline] bins insuficientes, usando mediana.")
        return np.full_like(f_seg, np.nanmedian(f_seg))

    s_val = fator_s * len(bin_t) * np.nanvar(bin_f) if fator_s > 0 else 0.0
    spline = UnivariateSpline(bin_t, bin_f, k=k, s=s_val)
    return spline(t_seg)


def fitting_spline(t_seg, f_seg, mask_seg, nbins, k, fator_s,
                   sigma_clip_interno=2.2, clip_iters=5):
    """
    Spline ajustada sobre mediana dos bins dos pontos bons.
 
    sigma_clip_interno : clip ANTES de binar para remover flares residuais
                         que a mask_good_flares não pegou.
                         Reduzir esse valor desce o modelo em regiões ruidosas.
                         None = desativa.
    """
    t_ok = t_seg[mask_seg]
    f_ok = f_seg[mask_seg]
 
    # remove flares residuais antes de binar
    if sigma_clip_interno is not None and len(f_ok) > k + 2:
        cl   = sigma_clip(f_ok, sigma=sigma_clip_interno, maxiters=clip_iters)
        t_ok = t_ok[~cl.mask]
        f_ok = f_ok[~cl.mask]
 
    if len(t_ok) < k + 2:
        print("    [spline] pontos insuficientes, usando mediana.")
        return np.full_like(f_seg, np.nanmedian(f_seg))
 
    edges   = np.linspace(t_ok.min(), t_ok.max(), nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
 
    bin_t, bin_f = [], []
    for i in range(nbins):
        in_bin = (t_ok >= edges[i]) & (t_ok < edges[i + 1])
        if np.sum(in_bin) >= 1:
            bin_t.append(centers[i])
            bin_f.append(np.nanmedian(f_ok[in_bin]))
 
    bin_t = np.array(bin_t)
    bin_f = np.array(bin_f)
 
    if len(bin_t) < k + 2:
        print("    [spline] bins insuficientes, usando mediana.")
        return np.full_like(f_seg, np.nanmedian(f_seg))
 
    s_val  = fator_s * len(bin_t) * np.nanvar(bin_f) if fator_s > 0 else 0.0
    spline = UnivariateSpline(bin_t, bin_f, k=k, s=s_val)
    return spline(t_seg)
 

def costurar_bordas(t, modelo, bordas, tamanho_janela=30):
    modelo_costurado = np.copy(modelo)
    print(f"\nAplicando costura em {len(bordas)} bordas...")
    for borda_idx in bordas:
        inicio = max(0, borda_idx - tamanho_janela)
        fim = min(len(t) - 1, borda_idx + tamanho_janela)
        if fim <= inicio:
            continue
        fluxo_A = modelo[inicio]
        fluxo_B = modelo[fim]
        tempo_A = t[inicio]
        tempo_B = t[fim]
        tempos = t[inicio:fim + 1]
        transicao = np.interp(tempos, [tempo_A, tempo_B], [fluxo_A, fluxo_B])
        modelo_costurado[inicio:fim + 1] = transicao
    return modelo_costurado

# ============================================================
# AJUSTE AUTOMÁTICO BASE
# ============================================================

modelo_manchas = np.zeros_like(f)
bordas_indices = set()
edges_auto = np.linspace(t.min(), t.max(), N_SEGMENTOS_AUTO + 1)
segmentos = [(edges_auto[i], edges_auto[i + 1]) for i in range(N_SEGMENTOS_AUTO)]

print("Ajuste automático...")
for i, (ini, fim) in enumerate(segmentos):
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        continue
    modelo_manchas[idx] = fitting_segment(
        t[idx], f[idx], mask_good_flares[idx], GRAU_AUTO, SIGMA_AUTO
    )
    if i > 0:
        bordas_indices.add(np.where(idx)[0][0])

# ============================================================
# AJUSTES MANUAIS (poly / flatten)
# ============================================================

print(f"Aplicando {len(ajustes_manuais)} ajustes manuais...")
for ini, fim, metodo, grau, sig in ajustes_manuais:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        continue

    if metodo == "poly":
        modelo_manchas[idx] = fitting_segment(
            t[idx], f[idx], mask_good_flares[idx], grau, sig
        )

    elif metodo == "flatten":
        print(f"  FLATTEN (global) {ini:.4f} - {fim:.4f}")
        modelo_manchas[idx] = trend.flux.value[idx]

        idx_inicio = np.where(idx)[0][0]
        janela = 15
        i0 = max(0, idx_inicio - janela)
        i1 = min(len(modelo_manchas) - 1, idx_inicio + janela)
        modelo_manchas[i0:i1 + 1] = np.linspace(
            modelo_manchas[i0], modelo_manchas[i1], i1 - i0 + 1
        )

    bordas_indices.add(np.where(idx)[0][0])
    bordas_indices.add(np.where(idx)[0][-1])

# ============================================================
# AJUSTES FLATTEN LOCAL POR REGIÃO (configuração completa por [t_i, t_f])
# ============================================================

print(f"\nAplicando {len(regioes_flatten)} região(ões) flatten local...")
for ini, fim, wlen, pord, sig, btol, niter in regioes_flatten:
    regiao = (t >= ini) & (t <= fim)
    if not np.any(regiao):
        print(f"  [flatten] nenhum ponto em {ini:.4f} - {fim:.4f}")
        continue

    # True = ignora no flatten. Usa somente pontos bons dentro da região.
    mask_flatten_regiao = (~regiao) | (~mask_good_flares)

    print(
        f"  [flatten] {ini:.4f} - {fim:.4f} | "
        f"window={wlen}, poly={pord}, sigma={sig}, "
        f"break_tol={btol}, niters={niter}"
    )

    flcd_r, trend_r = lc2_m.flatten(
        window_length=int(wlen),
        polyorder=int(pord),
        return_trend=True,
        break_tolerance=float(btol),
        niters=int(niter),
        sigma=float(sig),
        mask=mask_flatten_regiao
    )

    modelo_manchas[regiao] = trend_r.flux.value[regiao]
    bordas_indices.add(np.where(regiao)[0][0])
    bordas_indices.add(np.where(regiao)[0][-1])

# ============================================================
# AJUSTES SPLINE POR REGIÃO (aplicados por último)
# ============================================================


# ============================================================
# COSTURA FINAL
# ============================================================

modelo_manchas_suave = costurar_bordas(
    t, modelo_manchas, sorted(bordas_indices), tamanho_janela=30
)

# ============================================================
# RESÍDUO
# ============================================================

residual_manchas = f / modelo_manchas_suave
print("\nOK — Ajuste concluído.")

# ============================================================
# GRÁFICOS
# ============================================================

%matplotlib qt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados')
ax1.plot(t, modelo_manchas_suave, 'r-', lw=2.5, label='Modelo')

for ini, fim, *_ in regioes_spline:
    ax1.axvspan(ini, fim, color='dodgerblue', alpha=0.10, label='_spline')
for ini, fim, *_ in regioes_flatten:
    ax1.axvspan(ini, fim, color='mediumseagreen', alpha=0.10, label='_flatten')

ax1.set_ylabel("Fluxo")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7)
ax2.axhline(1, ls='--', alpha=0.5)

for ini, fim, *_ in regioes_spline:
    ax2.axvspan(ini, fim, color='dodgerblue', alpha=0.10)
for ini, fim, *_ in regioes_flatten:
    ax2.axvspan(ini, fim, color='mediumseagreen', alpha=0.10)

ax2.set_ylabel("Residual")
ax2.set_xlabel("Tempo [BTJD]")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


AJUSTE: Polinomial com Segmentos + Spline em Regiões
Ajuste automático...
Aplicando 7 ajustes manuais...

Aplicando 0 região(ões) flatten local...

Aplicando costura em 50 bordas...

OK — Ajuste concluído.


In [132]:
from astropy.stats import sigma_clip
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from lightkurve import LightCurve

# ============================================================
# PASSO 3: AJUSTE POLINOMIAL + FLATTEN LOCAL + SPLINE LOCAL
# ============================================================

print("\n" + "="*60)
print("AJUSTE: Polinomial com Segmentos + Spline em Regiões")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# ============================================================
# REGIÕES MANUAIS (poly / flatten)
# ============================================================

ajustes_manuais = [
    [3883.0050, 3883.4554, "poly", 4, 2.5],
    [3883.1409, 3883.7849, "poly", 4, 2.5],
    [3883.8264, 3884.0955, "poly", 4, 2.5],
    [3884.2828, 3885.3000, "poly", 4, 2.5],
    [3885.2839, 3885.4326, "poly", 4, 2.5],
    [3885.6205, 3885.7794, "poly", 2, 2.5],
    [3892.6280, 3892.7040, "poly", 1, 2.5],
]

# ============================================================
# REGIÕES SPLINE
#
# Formato: [t_ini, t_fim, nbins, k, fator_s, sigma_clip_int]
#
#   nbins          : bins para mediana (mais = mais aderente)
#   k              : grau da spline (3=cúbica, 4=quártica, 5=quíntica)
#   fator_s        : suavização (menor = mais colado; 0.0 = exato nos bins)
#   sigma_clip_int : clip aplicado ANTES de binar para remover flares
#                    residuais. Reduzir (ex: 1.8) desce mais o modelo.
#                    None = desativa o clip interno.
# ============================================================

regioes_spline = [
    # [t_ini,      t_fim,   nbins, k, fator_s, sigma_clip_int]
    [3899.86607, 3900.8691,  150,  4, 0.0001,  2.2],  # região com flares
    [3900.8273,  3901.44,     60,  3, 0.001,   2.2],  # região mais tranquila
    # [3905.0,   3907.00,     60,  3, 0.01,    2.2],  # descomente para mais
]

# ============================================================
# REGIÕES FLATTEN LOCAL
# Formato: [t_ini, t_fim, window_length, polyorder, sigma, break_tol, niters]
# ============================================================

regioes_flatten = [
    # [3900.5909, 3900.9512, 280, 1, 1.5, 1, 4],
]

# ============================================================
# FLATTEN GLOBAL
# ============================================================

flcd, trend = lc2_m.flatten(
    window_length=320,
    polyorder=3,
    return_trend=True,
    break_tolerance=10,
    niters=4,
    sigma=2.5,
    mask=mask_good_flares
)

# ============================================================
# AJUSTE AUTOMÁTICO
# ============================================================

N_SEGMENTOS_AUTO = 40
GRAU_AUTO        = 4
SIGMA_AUTO       = 3.0

# ============================================================
# FUNÇÕES
# ============================================================

def fitting_segment(t_seg, f_seg, mask_seg, deg, sigma):
    if len(t_seg) < deg + 2:
        return np.full_like(f_seg, np.nanmedian(f_seg))
    t_mid  = np.median(t_seg)
    t_s    = t_seg - t_mid
    good   = mask_seg.copy()
    modelo = np.full_like(f_seg, np.nan)
    for _ in range(5):
        if np.sum(good) < deg + 2:
            break
        coef        = np.polyfit(t_s[good], f_seg[good], deg=deg)
        modelo_good = np.polyval(coef, t_s[good])
        modelo      = np.interp(t_s, t_s[good], modelo_good)
        resid       = f_seg - modelo
        clipped     = sigma_clip(resid[good], sigma=sigma, maxiters=1)
        if clipped.mask is np.ma.nomask:
            break
        good[np.where(good)[0]] = ~clipped.mask
    if np.all(np.isnan(modelo)):
        modelo = np.full_like(f_seg, np.nanmedian(f_seg))
    return modelo


def fitting_spline(t_seg, f_seg, mask_seg, nbins, k, fator_s,
                   sigma_clip_interno=2.2, clip_iters=5):
    """
    Spline ajustada sobre mediana dos bins dos pontos bons.

    sigma_clip_interno : clip ANTES de binar para remover flares residuais
                         que a mask_good_flares não pegou.
                         Reduzir esse valor desce o modelo em regiões ruidosas.
                         None = desativa.
    """
    t_ok = t_seg[mask_seg]
    f_ok = f_seg[mask_seg]

    # remove flares residuais antes de binar
    if sigma_clip_interno is not None and len(f_ok) > k + 2:
        cl   = sigma_clip(f_ok, sigma=sigma_clip_interno, maxiters=clip_iters)
        t_ok = t_ok[~cl.mask]
        f_ok = f_ok[~cl.mask]

    if len(t_ok) < k + 2:
        print("    [spline] pontos insuficientes, usando mediana.")
        return np.full_like(f_seg, np.nanmedian(f_seg))

    edges   = np.linspace(t_ok.min(), t_ok.max(), nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    bin_t, bin_f = [], []
    for i in range(nbins):
        in_bin = (t_ok >= edges[i]) & (t_ok < edges[i + 1])
        if np.sum(in_bin) >= 1:
            bin_t.append(centers[i])
            bin_f.append(np.nanmedian(f_ok[in_bin]))

    bin_t = np.array(bin_t)
    bin_f = np.array(bin_f)

    if len(bin_t) < k + 2:
        print("    [spline] bins insuficientes, usando mediana.")
        return np.full_like(f_seg, np.nanmedian(f_seg))

    s_val  = fator_s * len(bin_t) * np.nanvar(bin_f) if fator_s > 0 else 0.0
    spline = UnivariateSpline(bin_t, bin_f, k=k, s=s_val)
    return spline(t_seg)


def costurar_bordas(t, modelo, bordas, tamanho_janela=30):
    modelo_costurado = np.copy(modelo)
    print(f"\nAplicando costura em {len(bordas)} bordas...")
    for borda_idx in bordas:
        inicio = max(0, borda_idx - tamanho_janela)
        fim    = min(len(t) - 1, borda_idx + tamanho_janela)
        if fim <= inicio:
            continue
        tempos    = t[inicio:fim + 1]
        transicao = np.interp(tempos, [t[inicio], t[fim]],
                              [modelo[inicio], modelo[fim]])
        modelo_costurado[inicio:fim + 1] = transicao
    return modelo_costurado

# ============================================================
# AJUSTE AUTOMÁTICO BASE
# ============================================================

modelo_manchas = np.zeros_like(f)
bordas_indices = set()
edges_auto     = np.linspace(t.min(), t.max(), N_SEGMENTOS_AUTO + 1)
segmentos      = [(edges_auto[i], edges_auto[i + 1]) for i in range(N_SEGMENTOS_AUTO)]

print("Ajuste automático...")
for i, (ini, fim) in enumerate(segmentos):
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        continue
    modelo_manchas[idx] = fitting_segment(
        t[idx], f[idx], mask_good_flares[idx], GRAU_AUTO, SIGMA_AUTO
    )
    if i > 0:
        bordas_indices.add(np.where(idx)[0][0])

# ============================================================
# AJUSTES MANUAIS (poly / flatten)
# ============================================================

print(f"Aplicando {len(ajustes_manuais)} ajustes manuais...")
for ini, fim, metodo, grau, sig in ajustes_manuais:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        continue
    if metodo == "poly":
        modelo_manchas[idx] = fitting_segment(
            t[idx], f[idx], mask_good_flares[idx], grau, sig
        )
    elif metodo == "flatten":
        print(f"  FLATTEN (global) {ini:.4f} - {fim:.4f}")
        modelo_manchas[idx] = trend.flux.value[idx]
        idx_inicio = np.where(idx)[0][0]
        i0 = max(0, idx_inicio - 15)
        i1 = min(len(modelo_manchas) - 1, idx_inicio + 15)
        modelo_manchas[i0:i1 + 1] = np.linspace(
            modelo_manchas[i0], modelo_manchas[i1], i1 - i0 + 1
        )
    bordas_indices.add(np.where(idx)[0][0])
    bordas_indices.add(np.where(idx)[0][-1])

# ============================================================
# AJUSTES FLATTEN LOCAL POR REGIÃO
# ============================================================

print(f"\nAplicando {len(regioes_flatten)} região(ões) flatten local...")
for ini, fim, wlen, pord, sig, btol, niter in regioes_flatten:
    regiao = (t >= ini) & (t <= fim)
    if not np.any(regiao):
        print(f"  [flatten] nenhum ponto em {ini:.4f} - {fim:.4f}")
        continue
    mask_flatten_regiao = (~regiao) | (~mask_good_flares)
    print(f"  [flatten] {ini:.4f} - {fim:.4f} | window={wlen}, poly={pord}, "
          f"sigma={sig}, break_tol={btol}, niters={niter}")
    flcd_r, trend_r = lc2_m.flatten(
        window_length=int(wlen), polyorder=int(pord),
        return_trend=True, break_tolerance=float(btol),
        niters=int(niter), sigma=float(sig),
        mask=mask_flatten_regiao
    )
    modelo_manchas[regiao] = trend_r.flux.value[regiao]
    bordas_indices.add(np.where(regiao)[0][0])
    bordas_indices.add(np.where(regiao)[0][-1])

# ============================================================
# AJUSTES SPLINE POR REGIÃO (aplicados por último)
# ============================================================

print(f"\nAplicando {len(regioes_spline)} região(ões) spline...")
for ini, fim, nbins, k, fator_s, sc_int in regioes_spline:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        print(f"  [spline] nenhum ponto em {ini:.4f} - {fim:.4f}")
        continue
    n_pts = np.sum(idx)
    print(f"  [spline] {ini:.4f} - {fim:.4f} | {n_pts} pts | "
          f"nbins={nbins}, k={k}, s={fator_s}, sigma_clip_int={sc_int}")
    modelo_manchas[idx] = fitting_spline(
        t[idx], f[idx], mask_good_flares[idx],
        nbins, k, fator_s,
        sigma_clip_interno=sc_int   # <-- passado direto da lista
    )
    bordas_indices.add(np.where(idx)[0][0])
    bordas_indices.add(np.where(idx)[0][-1])

# ============================================================
# COSTURA FINAL
# ============================================================

modelo_manchas_suave = costurar_bordas(
    t, modelo_manchas, sorted(bordas_indices), tamanho_janela=30
)

# ============================================================
# RESÍDUO
# ============================================================

residual_manchas = f / modelo_manchas_suave
print("\nOK — Ajuste concluído.")

# ============================================================
# GRÁFICOS
# ============================================================

%matplotlib qt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados')
ax1.plot(t, modelo_manchas_suave, 'r-', lw=2.5, label='Modelo')

for ini, fim, *_ in regioes_spline:
    ax1.axvspan(ini, fim, color='dodgerblue', alpha=0.10, label='_spline')
for ini, fim, *_ in regioes_flatten:
    ax1.axvspan(ini, fim, color='mediumseagreen', alpha=0.10, label='_flatten')

ax1.set_ylabel("Fluxo")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7)
ax2.axhline(1, ls='--', alpha=0.5)

for ini, fim, *_ in regioes_spline:
    ax2.axvspan(ini, fim, color='dodgerblue', alpha=0.10)
for ini, fim, *_ in regioes_flatten:
    ax2.axvspan(ini, fim, color='mediumseagreen', alpha=0.10)

ax2.set_ylabel("Residual")
ax2.set_xlabel("Tempo [BTJD]")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


AJUSTE: Polinomial com Segmentos + Spline em Regiões
Ajuste automático...
Aplicando 7 ajustes manuais...

Aplicando 0 região(ões) flatten local...

Aplicando 2 região(ões) spline...
  [spline] 3899.8661 - 3900.8691 | 522 pts | nbins=150, k=4, s=0.0001, sigma_clip_int=2.2
  [spline] 3900.8273 - 3901.4400 | 236 pts | nbins=60, k=3, s=0.001, sigma_clip_int=2.2

Aplicando costura em 54 bordas...

OK — Ajuste concluído.


In [101]:
%matplotlib qt
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# PARÂMETROS DOS PLANETAS (usa os já definidos; cria fallback se não existir)
# ============================================================

if "T0_b" not in globals():
    T0_b = 1330.39051
if "P_b" not in globals():
    P_b = 8.463000
if "dur_b" not in globals():
    dur_b = 3.50 / 24.0

if "T0_c" not in globals():
    T0_c = 1342.2223
if "P_c" not in globals():
    P_c = 18.859019
if "dur_c" not in globals():
    dur_c = 4.5 / 24.0

# ============================================================
# FUNÇÃO: centros de trânsito no intervalo de t
# ============================================================

def transit_times(T0, P, t_min, t_max):
    n_min = int(np.ceil((t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    return [T0 + n * P for n in range(n_min, n_max + 1)]

t_min, t_max = t.min(), t.max()
centers_b = transit_times(T0_b, P_b, t_min, t_max)
centers_c = transit_times(T0_c, P_c, t_min, t_max)

print(f"Transitos AU Mic b: {len(centers_b)}")
for tc in centers_b:
    print(f"  BTJD = {tc:.5f}")
print(f"Transitos AU Mic c: {len(centers_c)}")
for tc in centers_c:
    print(f"  BTJD = {tc:.5f}")

# ============================================================
# PEGAR PARÂMETROS DA CÉLULA 20 (se existirem)
# ============================================================

regioes_flatten_plot = globals().get("regioes_flatten", [])
regioes_spline_plot = globals().get("regioes_spline", [])
mascara_plot = globals().get("mascara_flares_list", [])

print(f"\nRegioes flatten da celula 20: {len(regioes_flatten_plot)}")
print(f"Regioes spline da celula 20:  {len(regioes_spline_plot)}")

# ============================================================
# FIGURA: 2 paineis
# ============================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# ---- Painel superior: dados + modelo ----
ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.5, label='Dados Originais', zorder=1)
ax1.plot(t, modelo_manchas_suave, 'r-', lw=2, label='Modelo (manchas)', zorder=3)

# Máscara de flares/trânsitos carregada no passo 2
_lm = False
for ini, fim in mascara_plot:
    lbl = 'Flares e Transitos detectados (TXT)' if not _lm else '_nolegend_'
    ax1.axvspan(ini, fim, color='orange', alpha=0.22, zorder=2, label=lbl)
    _lm = True

# Regioes FLATTEN da célula 20
_lf = False
for ini, fim, *_ in regioes_flatten_plot:
    lbl = 'Regiao flatten (celula 20)' if not _lf else '_nolegend_'
    ax1.axvspan(ini, fim, color='mediumseagreen', alpha=0.14, zorder=2, label=lbl)
    _lf = True

# Regioes SPLINE da célula 20
_ls = False
for ini, fim, *_ in regioes_spline_plot:
    lbl = 'Regiao spline (celula 20)' if not _ls else '_nolegend_'
    ax1.axvspan(ini, fim, color='dodgerblue', alpha=0.10, zorder=2, label=lbl)
    _ls = True

# Transitos AU Mic b
_lb = False
for tc in centers_b:
    lbl = 'Transito AU Mic b' if not _lb else '_nolegend_'
    ax1.axvspan(tc - dur_b / 2, tc + dur_b / 2, color='royalblue', alpha=0.22, zorder=2, label=lbl)
    ax1.axvline(tc, color='royalblue', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lb = True

# Transitos AU Mic c
_lc = False
for tc in centers_c:
    lbl = 'Transito AU Mic c' if not _lc else '_nolegend_'
    ax1.axvspan(tc - dur_c / 2, tc + dur_c / 2, color='seagreen', alpha=0.20, zorder=2, label=lbl)
    ax1.axvline(tc, color='seagreen', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lc = True

ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('AU Mic - Dados, Modelo e Parametros herdados da celula 20', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=10, framealpha=0.9, ncol=2)
ax1.grid(alpha=0.25)

# ---- Painel inferior: residual ----
ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.6, label='Residual', zorder=1)
ax2.axhline(1.0, color='gray', ls='--', lw=1.2, alpha=0.6)

_lm2 = False
for ini, fim in mascara_plot:
    lbl = 'Mascara' if not _lm2 else '_nolegend_'
    ax2.axvspan(ini, fim, color='orange', alpha=0.22, zorder=2, label=lbl)
    _lm2 = True

_lf2 = False
for ini, fim, *_ in regioes_flatten_plot:
    lbl = 'Flatten (celula 20)' if not _lf2 else '_nolegend_'
    ax2.axvspan(ini, fim, color='mediumseagreen', alpha=0.14, zorder=2, label=lbl)
    _lf2 = True

_ls2 = False
for ini, fim, *_ in regioes_spline_plot:
    lbl = 'Spline (celula 20)' if not _ls2 else '_nolegend_'
    ax2.axvspan(ini, fim, color='dodgerblue', alpha=0.10, zorder=2, label=lbl)
    _ls2 = True

ax2.set_ylabel('Fluxo Residual', fontsize=14)
ax2.set_xlabel('Tempo [BTJD dias]', fontsize=14)
ax2.legend(loc='upper right', fontsize=10, framealpha=0.9, ncol=2)
ax2.grid(alpha=0.25)

plt.tight_layout()
plt.show()

Transitos AU Mic b: 3
  BTJD = 3886.21651
  BTJD = 3894.67951
  BTJD = 3903.14251
Transitos AU Mic c: 2
  BTJD = 3888.18986
  BTJD = 3907.04888

Regioes flatten da celula 20: 0
Regioes spline da celula 20:  0
